In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Three-Way Merge Sort

We need to specify how three sorted lists $L_1$, $L_2$, and $L_3$ are merged in a way that the 
resulting list is also sorted.

 - If the list $L_1$ is empty, the result is the merge of $L_2$ and $L_3$: 
   $$ \mathtt{merge3}([], L_2, L_3) = \mathtt{merge}(L_2, L_3) $$
 - If the list $L_2$ is empty, the result is the merge of $L_1$ and $L_3$: 
   $$ \mathtt{merge3}(L_2, [], L_3) = \mathtt{merge}(L_1, L_3) $$
 - If the list $L_3$ is empty, the result is the merge of $L_1$ and $L_2$: 
   $$ \mathtt{merge3}(L_1, L_2, []) = \mathtt{merge}(L_1, L_2) $$
 - Otherwise, the lists $L_1$, $L_2$, $L_3$ must have the form 
   * $L_1 = [x_1] + R_1$,
   * $L_2 = [x_2] + R_2$, and
   * $L_3 = [x_3] + R_3$.
   
   Then there is a case distinction with respect to the minimum of $x_1$, $x_2$, and $x_3$:
   - $x_1 = \min(x_1, x_2, x_3) \rightarrow
     \mathtt{merge3}\bigl(L_1, L_2, L_3\bigr) = 
     \bigl[x_1\bigr] + \mathtt{merge}\bigl(R_1, L_2, L_3\bigr)$
   - $x_2 = \min(x_1, x_2, x_3) \rightarrow
     \mathtt{merge3}\bigl(L_1, L_2, L_3\bigr) = 
     \bigl[x_2\bigr] + \mathtt{merge}\bigl(L_1, R_2, L_3\bigr)$
   - $x_3 = \min(x_1, x_2, x_3) \rightarrow
     \mathtt{merge3}\bigl(L_1, L_2, L_3\bigr) = 
     \bigl[x_3\bigr] + \mathtt{merge}\bigl(L_1, L_2, R_3\bigr)$




In [ ]:
function merge(L1: number[], L2: number[]): number[] {
    if (L1.length === 0) return L2;
    if (L2.length === 0) return L1;

    const [x1, ...R1] = L1;
    const [x2, ...R2] = L2;

    if (x1 <= x2) {
        return [x1, ...merge(R1, L2)];
    } else {
        return [x2, ...merge(L1, R2)];
    }
}

In [ ]:
function merge3(L1: number[], L2: number[], L3: number[]): number[] {
    if (L1.length === 0) return merge(L2, L3);
    if (L2.length === 0) return merge(L1, L3);
    if (L3.length === 0) return merge(L1, L2);

    const [x1, ...R1] = L1;
    const [x2, ...R2] = L2;
    const [x3, ...R3] = L3;

    if (x1 <= x2) {
        if (x1 <= x3) {
            return [x1, ...merge3(R1, L2, L3)];
        } else { // x3 < x1
            return [x3, ...merge3(L1, L2, R3)];
        }
    } else { // x2 < x1
        if (x2 <= x3) {
            return [x2, ...merge3(L1, R2, L3)];
        } else { // x3 < x2
            return [x3, ...merge3(L1, L2, R3)];
        }
    }
}

In order to sort a list $L$ using <em style="color:blue;">merge sort</em> we proceed as follows:

 - If $L$ has less than two elements, then $L$ is already sorted.  Therefore we have: 
   $$ \#L < 2 \rightarrow \mathtt{sort}(L) = L $$
 - Otherwise, the list `L` is split into three lists that have approximately the same size.
  These lists are sorted recursively. Then, the sorted lists are merged in a way that the
  resulting list is sorted: 
  $$
  \#L \geq 2 \rightarrow \mathtt{sort}(L) =
       \mathtt{merge3}\Bigl(
            \mathtt{sort}\bigl(L.slice(0, \text{Math.floor}(L.length/3))\bigr),
            \mathtt{sort}\bigl(L.slice(\text{Math.floor}(L.length/3), \text{Math.floor}(2*L.length/3)))\bigr),
            \mathtt{sort}\bigl(L.slice(\text{Math.floor}(2*L.length/3)))\bigr)
       \Bigr)
  $$
  Here, `L.slice(0, Math.floor(L.length/3))` is the first part of the list,  
  `L.slice(Math.floor(L.length/3), Math.floor(2*L.length/3))` is the second part, and  
  `L.slice(Math.floor(2*L.length/3))` is the last part.


In [ ]:
function sort(L: number[]): number[] {
    const n = L.length;
    if (n < 2) return L;

    const L1 = L.slice(0, Math.floor(n / 3));
    const L2 = L.slice(Math.floor(n / 3), Math.floor(2 * n / 3));
    const L3 = L.slice(Math.floor(2 * n / 3));

    return merge3(sort(L1), sort(L2), sort(L3));
}

In [ ]:
sort([7, 8, 11, 12, 2, 5, 3, 7, 9, 3, 2])

## Testing

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [ ]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [ ]:
function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}

The function `isOrdered(L)` checks that the array `L` is sorted in ascending order.

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] > L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `sameElements(L, S)` returns `True`if the arrays `L` and `S` contain the same elements and, furthermore, each element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [ ]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function `randomIntRange(min, max)` generates a random integer within the range `[min, max)`. The lower bound `min` is included, while the upper bound `max` is excluded. Internally, it uses `Math.random()` to produce a uniformly distributed random value and scales it to the desired interval.

In [ ]:
function randomIntRange(min: number, max: number): number {
  return Math.floor(Math.random() * (max - min)) + min;
} 

The function $\texttt{testSort}(n, k)$ generates $n$ random arrays of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L = Array.from({ length: k }, () => randomIntRange(0, 2 * k));
    const S = sort(L);
    isOrdered(S);
    sameElements(L, S);
    process.stdout.write(".");
  }
  console.log("\nAll tests successful!");
}

In [ ]:
console.time("testSort");
testSort(100, 2000);
console.timeEnd("testSort")